# his-bench downstream results

- Main benchmark: task × decoder results table (UPerNet, MLPFusion), plus a separate UniverSat-B table
- A run only counts as complete once its log shows `Best ckpt path:` (printed after `trainer.test()`) — not SLURM's own exit status
- Keeps the best of multiple completed runs per (task, backbone, decoder), by test mIoU

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

## Configuration

Mirrors the exact naming scheme used by both `run_pangaea_consistency_*.sh` scripts (task/sensor map, weight_init tags, protocol overrides, experiment_setup suffix per decoder). If either script's naming changes, update it here too.

In [2]:
DISK_ROOT = Path("/dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/niklas/experiments/hsi-bench")

TASKS = ["cdl", "bdforet", "bnetd", "eurocrops", "nlcd", "treemap", "desis_cdl", "eo1_cdl", "oxhyperminerals_emit_l2a"]
TASK_LABELS = {
    "cdl": "CDL",
    "bdforet": "BD-F",
    "bnetd": "BNTD",
    "eurocrops": "EuCrops",
    "nlcd": "NLCD",
    "treemap": "TreeMap",
    "desis_cdl": "D-CDL",
    "eo1_cdl": "EO1-CDL",
    "oxhyperminerals_emit_l2a": "OxHyp",
}

BACKBONES = [
    "dofa_b", "dofav2_b", "copernicus_fm_b", "panopticon_b", "carl", "specaware_b",
    "vit_b", "vit_b_finetune", "vit_b_p8", "spec_vit_b_p8", "spec_vit_b_p16",
]
BACKBONE_LABELS = {
    "dofa_b": "DOFA-B",
    "dofav2_b": "DOFAv2-B",
    "copernicus_fm_b": "Copernicus-FM-B",
    "panopticon_b": "Panopticon-B",
    "carl": "CARL",
    "specaware_b": "SpecAware-B",
    "vit_b": "ViT-B/16 (random, frozen)",
    "vit_b_finetune": "ViT-B/16 (random, supervised)",
    "vit_b_p8": "ViT-B/8 (random, frozen)",
    "spec_vit_b_p8": "SpecViT-B/8 (random, frozen)",
    "spec_vit_b_p16": "SpecViT-B/16 (random, frozen)",
}

# Source sensor per task -- must match task_sensor_map in run_pangaea_consistency_cdl.sh
# (oxhyperminerals_emit_l2a is a separate task, run by run_oxhyperminerals_{upernet,mlpfusion}.sh,
# but follows the exact same experiment_setup/path convention, so it slots in here directly.)
TASK_SENSOR = {
    "cdl": "enmap", "bdforet": "enmap", "bnetd": "enmap", "eurocrops": "enmap",
    "nlcd": "enmap", "treemap": "enmap", "desis_cdl": "desis", "eo1_cdl": "eo1",
    "oxhyperminerals_emit_l2a": "emit",
}

# weight_init tag per backbone -- must match weight_init_tag_map in both scripts
WEIGHT_INIT = {
    "dofa_b": "dofa_pretrained",
    "dofav2_b": "dofav2_pretrained",
    "copernicus_fm_b": "copernicus_fm_pretrained",
    "panopticon_b": "panopticon_pretrained",
    "carl": "carl_pretrained",
    "specaware_b": "specaware_pretrained",
    "vit_b": "random_init",
    "vit_b_finetune": "random_init_finetune",
    "vit_b_p8": "random_init",
    "spec_vit_b_p8": "random_init",
    "spec_vit_b_p16": "random_init",
}

# protocol label per backbone -- default "linear"; only vit_b_finetune differs
PROTOCOL = {"vit_b_finetune": "finetune"}
DEFAULT_PROTOCOL = "linear"

# decoder folder name -> experiment_setup suffix each script uses
# ("<task>_<suffix>"), e.g. "cdl_upernet_linear" / "cdl_mlpfusion_linear"
DECODER_SETUP_SUFFIX = {
    "upernet_head": "upernet_linear",
    "mlp_fusion_head": "mlpfusion_linear",
}
DECODER_LABELS = {
    "upernet_head": "UPerNet",
    "mlp_fusion_head": "MLPFusion (bilinear)",
}

## Result extraction

In [3]:
# oxhyperminerals_emit_l2a uses MultiLabelSegmentationModule (multi-hot mineral
# labels), not SemanticSegmentationModule -- it logs test_MeanIoU / test_OverallAccuracy
# instead of test_MulticlassJaccardIndex / test_MulticlassAccuracy. Each entry below is
# tried in order and the first that matches wins, so the single-label tasks keep using
# their existing metric names and OxHyp falls through to its own -- no per-task branching
# needed elsewhere. Note these aren't computed identically (per-class IoU/accuracy over
# multi-hot vs single-hot labels), close enough to sit in the same "mIoU"/"accuracy"
# columns for a cross-task overview, but not a bit-for-bit identical metric.
_METRIC_PATTERNS = {
    "mIoU": [
        re.compile(r"test_MulticlassJaccardIndex\s*\S*\s*([0-9]+\.[0-9]+)"),
        re.compile(r"test_MeanIoU\s*\S*\s*([0-9]+\.[0-9]+)"),
    ],
    "accuracy": [
        re.compile(r"test_MulticlassAccuracy\s*\S*\s*([0-9]+\.[0-9]+)"),
        re.compile(r"test_OverallAccuracy\s*\S*\s*([0-9]+\.[0-9]+)"),
    ],
    "loss": [
        re.compile(r"test_loss\s*\S*\s*([0-9]+\.[0-9]+)"),
    ],
}
_CKPT_PATTERN = re.compile(r"Best ckpt path:\s*(\S+)")
_EPOCH_PATTERN = re.compile(r"epoch_(\d+)\.ckpt")


def parse_run_log(log_path: Path) -> dict | None:
    """Parse one .submitit/*/*_0_log.out file. Returns None if the run never
    reached trainer.test() (i.e. training didn't actually finish), regardless
    of what SLURM's own exit status says."""
    text = log_path.read_text(errors="ignore")
    if "Best ckpt path:" not in text:
        return None

    result: dict = {"log_path": str(log_path)}
    for name, patterns in _METRIC_PATTERNS.items():
        matches = None
        for pattern in patterns:
            found = pattern.findall(text)
            if found:
                matches = found
                break
        if not matches:
            return None  # completion line present but metrics missing -- treat as unusable
        result[name] = float(matches[-1])

    ckpt_match = _CKPT_PATTERN.search(text)
    if ckpt_match:
        result["checkpoint"] = ckpt_match.group(1)
        epoch_match = _EPOCH_PATTERN.search(ckpt_match.group(1))
        result["best_epoch"] = int(epoch_match.group(1)) if epoch_match else None
    return result


def find_best_result(task: str, backbone: str, decoder: str) -> dict | None:
    """Find every completed run for (task, backbone, decoder) and return the
    one with the highest test mIoU. There can legitimately be more than one
    multirun directory here (a rerun after a timeout, or an accidental
    duplicate submission) -- picking the best rather than e.g. the most
    recent avoids silently reporting a stale/failed rerun."""
    experiment_setup = f"{task}_{DECODER_SETUP_SUFFIX[decoder]}"
    sensor = TASK_SENSOR[task]
    weight_init = WEIGHT_INIT[backbone]
    protocol = PROTOCOL.get(backbone, DEFAULT_PROTOCOL)

    task_dir = (
        DISK_ROOT / "downstream" / experiment_setup / sensor / decoder
        / backbone / protocol / weight_init
    )
    log_files = sorted(task_dir.glob("multiruns/*/.submitit/*/*_0_log.out"))

    candidates = [parsed for f in log_files if (parsed := parse_run_log(f)) is not None]
    if not candidates:
        return None
    return max(candidates, key=lambda c: c["mIoU"])

## Build the long-format results table

One row per (decoder, task, backbone) with every metric -- the source of truth the pivoted tables below are built from. Also useful on its own for spotting which combinations are still missing/running.

In [4]:
rows = []
for decoder in DECODER_SETUP_SUFFIX:
    for task in TASKS:
        for backbone in BACKBONES:
            result = find_best_result(task, backbone, decoder)
            rows.append({
                "decoder": DECODER_LABELS[decoder],
                "task": TASK_LABELS[task],
                "backbone": BACKBONE_LABELS[backbone],
                "mIoU": result["mIoU"] if result else None,
                "accuracy": result["accuracy"] if result else None,
                "loss": result["loss"] if result else None,
                "best_epoch": result["best_epoch"] if result else None,
                "status": "done" if result else "missing/running",
            })

results_long = pd.DataFrame(rows)

missing = results_long[results_long["status"] == "missing/running"]
print(f"{len(results_long) - len(missing)} / {len(results_long)} combinations completed.")
if len(missing):
    print("Missing or still running:")
    display(missing[["decoder", "task", "backbone"]])

198 / 198 combinations completed.


## Pivoted tables: task (columns) x encoder (rows)

Test mIoU, best result per cell. Bold + shaded = best encoder for that task.

In [5]:
def pivot_table(decoder: str, metric: str = "mIoU") -> pd.DataFrame:
    subset = results_long[results_long["decoder"] == DECODER_LABELS[decoder]]
    table = subset.pivot(index="backbone", columns="task", values=metric)
    row_order = [BACKBONE_LABELS[b] for b in BACKBONES]
    col_order = [TASK_LABELS[t] for t in TASKS]
    return table.reindex(index=row_order, columns=col_order)


def style_table(df: pd.DataFrame, caption: str):
    return (
        df.style
        .format("{:.3f}", na_rep="\u2014")
        .highlight_max(axis=0, props="font-weight:bold;background-color:#e6f4ea;")
        .set_caption(caption)
    )

In [6]:
upernet_table = pivot_table("upernet_head")
style_table(upernet_table, "UPerNet -- test mIoU")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL,OxHyp
backbone,,,,,,,,,
DOFA-B,0.530,0.459,0.347,0.419,0.301,0.324,0.534,0.572,0.510
DOFAv2-B,0.501,0.451,0.345,0.420,0.295,0.316,0.515,0.573,0.476
Copernicus-FM-B,0.528,0.451,0.347,0.422,0.295,0.321,0.526,0.573,0.499
Panopticon-B,0.587,0.583,0.378,0.478,0.382,0.387,0.560,0.624,0.512
CARL,0.593,0.570,0.390,0.501,0.350,0.368,0.613,0.572,0.564
SpecAware-B,0.535,0.450,0.342,0.436,0.305,0.306,0.482,0.553,0.347
"ViT-B/16 (random, frozen)",0.548,0.521,0.352,0.449,0.316,0.364,0.529,0.551,0.595
"ViT-B/16 (random, supervised)",0.683,0.661,0.394,0.548,0.411,0.410,0.613,0.636,0.715
"ViT-B/8 (random, frozen)",0.595,0.549,0.381,0.485,0.336,0.367,0.565,0.578,0.578


In [7]:
mlp_fusion_table = pivot_table("mlp_fusion_head")
style_table(mlp_fusion_table, "MLPFusion (bilinear) -- test mIoU")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL,OxHyp
backbone,,,,,,,,,
DOFA-B,0.580,0.495,0.356,0.452,0.348,0.355,0.573,0.570,0.612
DOFAv2-B,0.515,0.468,0.354,0.429,0.347,0.345,0.549,0.557,0.571
Copernicus-FM-B,0.571,0.468,0.356,0.449,0.348,0.353,0.563,0.569,0.597
Panopticon-B,0.622,0.621,0.387,0.507,0.399,0.396,0.586,0.631,0.630
CARL,0.599,0.544,0.387,0.488,0.374,0.374,0.614,0.521,0.624
SpecAware-B,0.490,0.407,0.317,0.418,0.326,0.294,0.425,0.417,0.335
"ViT-B/16 (random, frozen)",0.391,0.408,0.298,0.349,0.314,0.318,0.363,0.376,0.582
"ViT-B/16 (random, supervised)",0.573,0.571,0.367,0.468,0.398,0.394,0.540,0.548,0.634
"ViT-B/8 (random, frozen)",0.408,0.419,0.314,0.369,0.338,0.326,0.366,0.375,0.594


## Ranking: how each encoder stacks up per task

Per-task rank of each backbone (1 = best mIoU on that task), plus an **Avg Rank** column (mean rank across all tasks) to summarize overall standing across the whole benchmark. Sorted by Avg Rank, so the strongest encoder overall is at the top. A missing/still-running combination is left unranked (`—`), not silently dropped or treated as worst.

In [8]:
def rank_table(decoder: str, metric: str = "mIoU", higher_is_better: bool = True) -> pd.DataFrame:
    """Per-task rank of each backbone (1 = best), plus an Avg Rank column
    (mean rank across tasks) summarizing overall standing. Ties share the
    same (minimum) rank. Missing/still-running combinations stay NaN --
    excluded from Avg Rank via pandas' default skipna, not counted as a
    last-place finish."""
    values = pivot_table(decoder, metric=metric)
    ranks = values.rank(axis=0, ascending=not higher_is_better, method="min")
    ranks["Avg Rank"] = ranks.mean(axis=1).round(2)
    return ranks.sort_values("Avg Rank")


def style_rank_table(df: pd.DataFrame, caption: str):
    task_cols = [c for c in df.columns if c != "Avg Rank"]
    n_backbones = len(BACKBONES)
    return (
        df.style
        .format("{:.0f}", subset=task_cols, na_rep="—")
        .format("{:.2f}", subset=["Avg Rank"], na_rep="—")
        .background_gradient(cmap="RdYlGn_r", subset=task_cols, vmin=1, vmax=n_backbones)
        .background_gradient(cmap="RdYlGn_r", subset=["Avg Rank"], vmin=1, vmax=n_backbones)
        .set_caption(caption)
    )

In [9]:
upernet_ranks = rank_table("upernet_head")
style_rank_table(upernet_ranks, "UPerNet -- rank per task (1 = best), sorted by Avg Rank")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL,OxHyp,Avg Rank
backbone,,,,,,,,,,
"ViT-B/16 (random, supervised)",1,1,2,1,1,1,1,1,1,1.11
"SpecViT-B/8 (random, frozen)",2,2,1,2,3,2,3,3,3,2.33
Panopticon-B,6,3,5,6,2,3,5,2,7,4.33
CARL,5,5,3,3,4,5,2,8,5,4.44
"ViT-B/8 (random, frozen)",4,6,4,5,5,6,4,4,4,4.67
"SpecViT-B/16 (random, frozen)",3,4,6,4,6,4,6,9,6,5.33
"ViT-B/16 (random, frozen)",7,7,7,7,7,7,8,11,2,7.00
DOFA-B,9,8,9,11,9,8,7,7,8,8.44
Copernicus-FM-B,10,10,8,9,11,9,9,5,9,8.89


In [10]:
mlp_fusion_ranks = rank_table("mlp_fusion_head")
style_rank_table(mlp_fusion_ranks, "MLPFusion (bilinear) -- rank per task (1 = best), sorted by Avg Rank")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL,OxHyp,Avg Rank
backbone,,,,,,,,,,
Panopticon-B,1,1,2,2,1,1,2,1,2,1.44
CARL,3,5,1,3,3,3,1,7,3,3.22
"ViT-B/16 (random, supervised)",5,3,4,5,2,2,7,6,1,3.89
"SpecViT-B/8 (random, frozen)",2,2,3,1,4,4,8,5,8,4.11
DOFA-B,4,6,6,6,5,6,3,2,4,4.67
Copernicus-FM-B,6,8,5,7,6,7,4,3,6,5.78
"SpecViT-B/16 (random, frozen)",7,4,8,4,8,5,5,8,5,6.00
DOFAv2-B,8,7,7,8,7,8,6,4,10,7.22
"ViT-B/8 (random, frozen)",10,9,10,10,9,9,10,11,7,9.44


## UniverSat-B (ConvSegmentor) — its own table

UniverSat-B doesn't fit the UPerNet/MLPFusion comparison above — it has a single dense-feature-grid
output (no per-block taps), so it runs through a completely different decoder path (`ConvSegmentor` +
`universat_conv_head`, see `scripts/downstream/run_oxhyperminerals_universat.sh` /
`run_universat_convsegmentor.sh`) and its own native image size per sensor (64px for EMIT, 128px for
EnMAP/DESIS/EO-1, not resized to 224 like the other backbones). Same standard full-band setup, frozen
backbone / linear probe, as the rest of this notebook — kept as a separate lookup (`{task}_convseg_linear`
experiment_setup, `universat_b`/`universat_conv_head`) rather than folded into `BACKBONES`/
`DECODER_SETUP_SUFFIX` above, since iterating it through the same loop as the other 11 backbones would
report every (backbone, `universat_conv_head`) combination that was never run as "missing".

`nlcd`/`treemap` were dropped for UniverSat specifically (too slow relative to the rest of this
benchmark, same tradeoff made throughout this project) — left blank (`—`), not silently omitted.

In [11]:
UNIVERSAT_BACKBONE = "universat_b"
UNIVERSAT_DECODER = "universat_conv_head"
UNIVERSAT_WEIGHT_INIT = "universat_pretrained"
UNIVERSAT_PROTOCOL = "linear"


def find_universat_result(task: str) -> dict | None:
    """Same lookup as find_best_result, but UniverSat's experiment_setup suffix
    ("convseg_linear") and single backbone/decoder pairing don't fit the
    DECODER_SETUP_SUFFIX/BACKBONES iteration above, so this stays standalone."""
    experiment_setup = f"{task}_convseg_linear"
    sensor = TASK_SENSOR[task]
    task_dir = (
        DISK_ROOT / "downstream" / experiment_setup / sensor / UNIVERSAT_DECODER
        / UNIVERSAT_BACKBONE / UNIVERSAT_PROTOCOL / UNIVERSAT_WEIGHT_INIT
    )
    log_files = sorted(task_dir.glob("multiruns/*/.submitit/*/*_0_log.out"))
    candidates = [parsed for f in log_files if (parsed := parse_run_log(f)) is not None]
    if not candidates:
        return None
    return max(candidates, key=lambda c: c["mIoU"])


universat_rows = []
for task in TASKS:
    result = find_universat_result(task)
    universat_rows.append({
        "task": TASK_LABELS[task],
        "mIoU": result["mIoU"] if result else None,
        "accuracy": result["accuracy"] if result else None,
        "loss": result["loss"] if result else None,
        "status": "done" if result else "missing/running",
    })

universat_long = pd.DataFrame(universat_rows)

missing_u = universat_long[universat_long["status"] == "missing/running"]
print(f"{len(universat_long) - len(missing_u)} / {len(universat_long)} tasks completed for UniverSat-B.")
if len(missing_u):
    print("Missing or still running:")
    display(missing_u[["task"]])

universat_table = universat_long.set_index("task")[["mIoU"]].reindex(
    [TASK_LABELS[t] for t in TASKS]
).T
universat_table.index = ["UniverSat-B"]

(
    universat_table.style
    .format("{:.3f}", na_rep="—")
    .set_caption("UniverSat-B (ConvSegmentor) -- test mIoU")
)

7 / 9 tasks completed for UniverSat-B.
Missing or still running:


,task
4,NLCD
5,TreeMap


task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL,OxHyp
UniverSat-B,0.593,0.537,0.393,0.485,—,—,0.560,0.643,0.496


## Optional: same tables for accuracy / loss

Swap `metric=` below for anything in `results_long.columns` (`accuracy`, `loss`, `best_epoch`).

In [12]:
style_table(pivot_table("upernet_head", metric="accuracy"), "UPerNet -- test accuracy")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL,OxHyp
backbone,,,,,,,,,
DOFA-B,0.692,0.629,0.515,0.591,0.463,0.489,0.696,0.728,0.860
DOFAv2-B,0.668,0.622,0.513,0.592,0.455,0.480,0.680,0.729,0.850
Copernicus-FM-B,0.691,0.622,0.516,0.593,0.455,0.487,0.689,0.729,0.857
Panopticon-B,0.740,0.737,0.549,0.647,0.553,0.558,0.718,0.769,0.860
CARL,0.745,0.726,0.561,0.668,0.518,0.538,0.760,0.727,0.880
SpecAware-B,0.697,0.621,0.510,0.607,0.467,0.468,0.651,0.712,0.818
"ViT-B/16 (random, frozen)",0.708,0.685,0.521,0.620,0.480,0.534,0.692,0.710,0.885
"ViT-B/16 (random, supervised)",0.812,0.796,0.565,0.708,0.583,0.581,0.760,0.777,0.920
"ViT-B/8 (random, frozen)",0.746,0.709,0.552,0.653,0.503,0.537,0.722,0.733,0.886


In [13]:
style_table(pivot_table("mlp_fusion_head", metric="accuracy"), "MLPFusion (bilinear) -- test accuracy")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL,OxHyp
backbone,,,,,,,,,
DOFA-B,0.734,0.662,0.525,0.623,0.517,0.524,0.729,0.726,0.890
DOFAv2-B,0.680,0.637,0.523,0.601,0.515,0.513,0.709,0.715,0.877
Copernicus-FM-B,0.727,0.637,0.525,0.620,0.516,0.522,0.720,0.725,0.885
Panopticon-B,0.767,0.766,0.558,0.672,0.571,0.568,0.739,0.774,0.897
CARL,0.749,0.704,0.558,0.656,0.545,0.545,0.761,0.685,0.893
SpecAware-B,0.658,0.579,0.481,0.590,0.491,0.454,0.596,0.589,0.812
"ViT-B/16 (random, frozen)",0.562,0.579,0.460,0.517,0.478,0.482,0.532,0.547,0.882
"ViT-B/16 (random, supervised)",0.729,0.727,0.537,0.637,0.570,0.565,0.701,0.708,0.895
"ViT-B/8 (random, frozen)",0.579,0.591,0.478,0.539,0.505,0.492,0.536,0.545,0.884
